# Stage 8C.1 - Full Beam-to-Write Cockpit MVP

This notebook is an optical/energy/exposure cockpit. It does not predict material modification.

The purpose is to tune the editable source-to-sample lab chain, inspect real optical-field fluence diagnostics, and keep exposure bookkeeping visible while future material-response modules remain intentionally disabled.

In [ ]:
# ============================================================
# FULL BEAM-TO-WRITE COCKPIT MVP - USER CONTROLS
# ============================================================

planning_mode = True
save_outputs = False
figure_dpi = 180
show_caveats = True
show_warnings = True
show_diagnostic_panels = True

# --- Simulation quality ---
engine_preset = "fast"          # fast | balanced | publication
engine_path = "ideal"           # ideal | realistic
require_real_field = True
allow_synthetic_demo_field = False

# --- Laser source ---
wavelength_nm = 1030.0
pulse_duration_fs = 260.0
repetition_rate_Hz = 25_000.0
pulse_energy_before_optics_uJ = 200.0
average_power_limit_W = 10.0
beam_radius_mm = 2.0
polarisation_state = "linear"

# --- Pre-SLM beam conditioning ---
pre_slm_transmission = 1.0
input_beam_radius_mm = 2.0
beam_ellipticity = 1.0
pointing_offset_x_um = 0.0
pointing_offset_y_um = 0.0
aperture_radius_mm = None

# --- Telescope / beam expander ---
telescope_enabled = False
telescope_magnification = 1.0
telescope_transmission = 1.0

# --- SLM1 phase / vortex ---
slm1_enabled = True
phase_profile = "vortex"
ell = 3
phase_quantisation_levels = 256
slm_pixel_pitch_um = 8.0
slm_active_width_px = None
slm_active_height_px = None
slm1_diffraction_efficiency = 0.95

# --- SLM2 / axicon / holography ---
generation_method = "holographic"    # holographic | physical
target_core_diameter_um = 3.0
target_bessel_length_um = 150.0
axicon_k_r = None
blaze_period_px = 20
slm2_diffraction_efficiency = 0.95
slm2_conjugate_mode = "preserve_vortex"
physical_axicon_enabled = False
physical_axicon_angle_deg = None

# --- First-order filtering ---
first_order_filter_enabled = True
selected_first_order_fraction = 0.73
filter_radius_px_or_lpmm = None
zero_order_leakage_fraction = 0.0

# --- Relay optics ---
relay_transmission = 0.90
relay_magnification = 1.0
relay_aberration_enabled = False

# --- Objective and pupil ---
objective_NA = 0.45
objective_transmission = 0.85
objective_effective_focal_length_mm = None
objective_pupil_diameter_mm = None
pupil_diameter_override_mm = None

# --- Sample/interface ---
material_name = "Cr:ZnSe"
refractive_index = 2.44
sample_thickness_mm = 5.0
focus_depth_um = 100.0
sample_interface_transmission = 0.95
use_fresnel_interface_estimate = False
surface_tilt_mrad = 0.0

# --- In-sample propagation / field stack ---
grid_N = None
device_downsample = None
axial_planes = None
crop_window_um = None

# --- Field/fluence display ---
selected_z_um = "target_depth"       # target_depth | optical_peak | sample_surface | custom
selected_z_mode = selected_z_um
custom_z_um = 100.0
central_roi_half_width_um = 10.0
display_scaling = "percentile"       # linear | log | percentile
display_percentile_clip = (0.5, 99.5)
fluence_normalisation_mode = "per_plane_transverse_energy"

# --- Writing plan / exposure bookkeeping ---
writing_mode = "line_x"              # static_single | static_multi | line_x | line_y | line_z | tilted_line | disabled
scan_axis = "x"
scan_speed_mm_s = 1.0
line_length_um = 500.0
effective_diameter_um = 3.0
num_static_pulses = 100
num_passes = 1
z_step_um = 0.0
tilt_angle_deg = 0.0

# --- Future physics toggles: must remain disabled in Stage 8C.1 ---
enable_material_response = False
enable_threshold_proxy = False
enable_dose_accumulation = False
enable_nonlinear_proxy = False
enable_thermal_proxy = False
enable_microscope_proxy = False
enable_calibrated_prediction = False


## 1. Purpose and claim boundary

This notebook is an optical/energy/exposure cockpit. It does not predict material modification. It scales real optical-field intensity arrays to optical fluence, reports energy throughput, and performs exposure bookkeeping only.

In [ ]:
# Guard rails for saving, demo fields, and future physics toggles.
if save_outputs and not show_caveats:
    raise ValueError("Cannot save Stage 8C.1 outputs with show_caveats=False.")

future_flags = {
    "enable_material_response": enable_material_response,
    "enable_threshold_proxy": enable_threshold_proxy,
    "enable_dose_accumulation": enable_dose_accumulation,
    "enable_nonlinear_proxy": enable_nonlinear_proxy,
    "enable_thermal_proxy": enable_thermal_proxy,
    "enable_microscope_proxy": enable_microscope_proxy,
    "enable_calibrated_prediction": enable_calibrated_prediction,
}
if any(future_flags.values()):
    enabled = [name for name, value in future_flags.items() if value]
    raise NotImplementedError(f"Stage 8C.1 does not implement future physics toggles: {enabled}")
if allow_synthetic_demo_field and save_outputs:
    raise ValueError("unit_test_or_demo_only fields cannot be saved as governed outputs.")

CAVEAT_TEXT = (
    "This is an optical/energy/exposure cockpit. It does not model absorption, "
    "material modification, ablation, cracks, voids, waveguides, nonlinear propagation, "
    "or thermal accumulation. Energy-scaled fluence is optical fluence only."
)
if show_caveats:
    print(CAVEAT_TEXT)


In [ ]:
from pathlib import Path
import sys
import numpy as np

_root = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "bessel_twin_core.py").is_file():
        _root = _candidate
        break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from vbb_study.digital_twin.field_coupling import (
    MissingOpticalFieldError,
    extract_plane_from_surfacefield,
    extract_stack_from_surfacefield,
    plane_from_arrays,
)
from vbb_study.digital_twin.field_fluence import scale_plane_to_fluence, scale_stack_to_fluence
from vbb_study.digital_twin.lab_realism_controls import (
    build_energy_ledger_from_controls,
    build_exposure_summary_from_controls,
    build_lab_realism_report,
)
from vbb_study.digital_twin.cockpit_dashboard import (
    compute_peak_location_diagnostics,
    build_warning_flags,
    plot_integrated_cockpit_dashboard,
    make_interpretation_text,
)


## 2. User controls

All editable controls are in the first code cell.

In [ ]:
control_names = [
    "planning_mode", "save_outputs", "figure_dpi", "show_caveats", "show_warnings", "show_diagnostic_panels",
    "engine_preset", "engine_path", "require_real_field", "allow_synthetic_demo_field",
    "wavelength_nm", "pulse_duration_fs", "repetition_rate_Hz", "pulse_energy_before_optics_uJ", "average_power_limit_W", "beam_radius_mm", "polarisation_state",
    "pre_slm_transmission", "input_beam_radius_mm", "beam_ellipticity", "pointing_offset_x_um", "pointing_offset_y_um", "aperture_radius_mm",
    "telescope_enabled", "telescope_magnification", "telescope_transmission",
    "slm1_enabled", "phase_profile", "ell", "phase_quantisation_levels", "slm_pixel_pitch_um", "slm_active_width_px", "slm_active_height_px", "slm1_diffraction_efficiency",
    "generation_method", "target_core_diameter_um", "target_bessel_length_um", "axicon_k_r", "blaze_period_px", "slm2_diffraction_efficiency", "slm2_conjugate_mode", "physical_axicon_enabled", "physical_axicon_angle_deg",
    "first_order_filter_enabled", "selected_first_order_fraction", "filter_radius_px_or_lpmm", "zero_order_leakage_fraction",
    "relay_transmission", "relay_magnification", "relay_aberration_enabled",
    "objective_NA", "objective_transmission", "objective_effective_focal_length_mm", "objective_pupil_diameter_mm", "pupil_diameter_override_mm",
    "material_name", "refractive_index", "sample_thickness_mm", "focus_depth_um", "sample_interface_transmission", "use_fresnel_interface_estimate", "surface_tilt_mrad",
    "grid_N", "device_downsample", "axial_planes", "crop_window_um",
    "selected_z_um", "selected_z_mode", "custom_z_um", "central_roi_half_width_um", "display_scaling", "display_percentile_clip", "fluence_normalisation_mode",
    "writing_mode", "scan_axis", "scan_speed_mm_s", "line_length_um", "effective_diameter_um", "num_static_pulses", "num_passes", "z_step_um", "tilt_angle_deg",
    "enable_material_response", "enable_threshold_proxy", "enable_dose_accumulation", "enable_nonlinear_proxy", "enable_thermal_proxy", "enable_microscope_proxy", "enable_calibrated_prediction",
]
controls = {name: globals()[name] for name in control_names}


## 3. Stage-by-stage lab realism report

The `LabRealismReport` is the central source-to-sample table: every stage reports editable inputs, computed outputs, warnings, missing metrics, and handoff.

In [ ]:
ledger = build_energy_ledger_from_controls(controls)
exposure_summary = build_exposure_summary_from_controls(
    controls,
    pulse_energy_at_sample_uJ=ledger.energy_at_sample_uJ,
    repetition_rate_Hz=repetition_rate_Hz,
)
lab_report = build_lab_realism_report(controls, energy_ledger=ledger, exposure_summary=exposure_summary)
lab_report.to_dataframe()


## 4. Experiment request summary

In [ ]:
experiment_request = {
    "laser": f"{wavelength_nm} nm, {pulse_duration_fs} fs, {repetition_rate_Hz} Hz",
    "route": f"{generation_method} VBB, ell={ell}",
    "slm_settings": f"SLM1 {phase_profile}; SLM2 {slm2_conjugate_mode}; blaze={blaze_period_px} px",
    "optical_chain": f"first order={selected_first_order_fraction}, relay={relay_transmission}, objective={objective_transmission}",
    "sample": f"{material_name}, n={refractive_index}, focus depth={focus_depth_um} um",
    "writing_plan": f"{writing_mode}, {scan_speed_mm_s} mm/s, {line_length_um} um",
    "selected_visual_plane": selected_z_um,
}
experiment_request


## 5. Energy ledger

In [ ]:
print(f"Pulse energy before optics: {pulse_energy_before_optics_uJ:.4g} uJ")
print(f"Energy at sample: {ledger.energy_at_sample_uJ:.4g} uJ")
print(f"Average power at sample: {ledger.average_power_at_sample_W:.4g} W")
print(f"Cumulative throughput: {ledger.total_throughput_fraction:.2%}")
for row in ledger.rows:
    print(f"{row.component_name:30s} {row.energy_in_uJ:9.4g} -> {row.energy_out_uJ:9.4g} uJ  frac={row.fraction_this_component:.3g}")
for warning in ledger.ledger_warnings:
    print("WARNING:", warning)


## 6. Lab realism / hardware feasibility panel

Energy/power limit, first-order geometry, pupil fill/clipping, SLM sampling/downsampling, route validity, crop-window/captured-power drift, peak near edge, pulse overlap, and material-response availability are all shown as pass/caution/fail/missing/disabled states.

In [ ]:
warning_flags = build_warning_flags(energy_ledger=ledger, exposure_summary=exposure_summary)
warning_flags


## 7. Real optical field acquisition

Use the real repository engine. If no governed optical field is available and `require_real_field=True`, fail loudly. A labelled `unit_test_or_demo_only` branch exists only for local notebook wiring checks and refuses saving.

In [ ]:
surface_field = None
volume = None
field_source_status = "not loaded"

try:
    import bessel_twin_core as bt
    engine_result = bt.run_case(preset=engine_preset, path=engine_path, case_id="stage8c1_integrated_cockpit")
    surface_field = engine_result.get("surface_field")
    volume = engine_result.get("volume")
    field_source_status = "real_optical_field"
except Exception as exc:
    field_source_status = f"engine unavailable: {exc}"
    print(field_source_status)

if volume is None and require_real_field and not allow_synthetic_demo_field:
    raise MissingOpticalFieldError("No real optical volume available; refusing to fabricate a production field.")

if volume is None and allow_synthetic_demo_field:
    save_outputs = False
    print("unit_test_or_demo_only: synthetic demonstration field is labelled and cannot be saved.")
    n = 64
    z_demo = np.linspace(0.0, 150.0, 5)
    x_demo = (np.arange(n) - (n - 1) / 2.0) * 0.5
    X, Y = np.meshgrid(x_demo, x_demo, indexing="xy")
    stack_demo = []
    for zval in z_demo:
        R = np.hypot(X, Y)
        stack_demo.append(np.exp(-(R ** 2) / (2 * 3.0 ** 2)) * (1.0 + 0.1 * zval / max(z_demo)))
    volume = {"intensity_stack": np.asarray(stack_demo), "z": z_demo * 1e-6, "crop_grid": {"x": x_demo * 1e-6, "y": x_demo * 1e-6}}

stack = extract_stack_from_surfacefield(volume)
plane = extract_plane_from_surfacefield(surface_field, preferred_z_um=0.0) if surface_field is not None else None
print("field_source_status:", field_source_status)
print("stack:", stack.intensity_zyx.shape, "dx_um", stack.dx_um, "dy_um", stack.dy_um)


## 8. Field-to-fluence scaling

This section reports selected plane, central ROI peak, target-depth peak, sample-surface peak, peak intensity estimate, propagation/crop captured-power drift, global peak diagnostics, and edge/corner peak warning.

In [ ]:
fluence_stack = scale_stack_to_fluence(stack, ledger.energy_at_sample_uJ)
fluence_plane = scale_plane_to_fluence(plane, ledger.energy_at_sample_uJ) if plane is not None else None
diagnostics = compute_peak_location_diagnostics(
    stack,
    fluence_stack,
    target_depth_um=focus_depth_um,
    central_roi_half_width_um=central_roi_half_width_um,
    selected_z_mode=selected_z_mode,
    custom_z_um=custom_z_um,
    pulse_duration_fs=pulse_duration_fs,
)
field_summary = {
    "source_status": stack.source_status,
    "dx_um": stack.dx_um,
    "dy_um": stack.dy_um,
    "propagation_energy_drift_fraction": fluence_stack.propagation_energy_drift_fraction,
}
lab_report = build_lab_realism_report(
    controls,
    energy_ledger=ledger,
    exposure_summary=exposure_summary,
    field_summary=field_summary,
    diagnostics=diagnostics,
)
warning_flags = build_warning_flags(energy_ledger=ledger, exposure_summary=exposure_summary, diagnostics=diagnostics)
diagnostics


## 9. Exposure bookkeeping

This is exposure bookkeeping, not material response.

In [ ]:
for key, value in exposure_summary.items():
    print(f"{key}: {value}")


## 10. Main integrated cockpit dashboard figure

The integrated dashboard shows experiment summary, energy ledger, exposure bookkeeping, lab realism status, XY intensity, XY fluence, central ROI zoom, XZ fluence with surface/target/selected/peak markers, peak fluence by z, raw captured-power drift, warnings, caveats, and disabled future physics.

In [ ]:
output_path = _root / "outputs" / "figures" / "digital_twin" / "stage8c1_integrated_cockpit_preview.png"
fig = plot_integrated_cockpit_dashboard(
    stack,
    fluence_stack,
    controls=controls,
    energy_ledger=ledger,
    exposure_summary=exposure_summary,
    lab_report=lab_report,
    diagnostics=diagnostics,
    output_path=output_path if save_outputs else None,
    show_caveats=show_caveats,
    dpi=figure_dpi,
    display_scaling=display_scaling,
    display_percentile_clip=display_percentile_clip,
)
fig


## 11. Interpretation panel

In [ ]:
print(make_interpretation_text(
    controls=controls,
    energy_ledger=ledger,
    exposure_summary=exposure_summary,
    diagnostics=diagnostics,
    lab_report=lab_report,
))


## 12. Disabled future physics panels

Material response: disabled until calibration

Dose accumulation: Stage 8E

Threshold maps: future proxy/calibration stage

Microscope proxy: later

Waveguide prediction: later, requires dn calibration

Surface ablation: later, separate model

Nonlinear propagation: later, requires material constants

Thermal accumulation: later, requires thermal model/constants